# Closure Certificate vs a 4-Signal Confidence Battery — on NATURAL Re-DocRED Kinship Prose

**Artifact:** STEP-B (iter-7) of *"No Derivation, No Relation: A Closure Certificate for Knowledge Extraction"*.

This demo carries the load-bearing diagnostic off **templated** CLUTRR text onto **genuinely-natural
Wikipedia introductory prose** (the Re-DocRED / DocRED kinship corpus). It pits a **closure
CERTIFICATE** — a forward least-fixpoint UNION over LLM-extracted kinship edges that **abstains
when no derivation path exists** — against the strongest **confidence / uncertainty BATTERY** on
the raw LLM answerer:

| signal | meaning |
|---|---|
| `verbalized` | the answerer's self-reported confidence in [0,1] |
| `sc_margin`  | self-consistency vote-margin (top-relation fraction over k=10 samples) |
| `ptrue`      | Kadavath P(True): the model's probability its own answer is correct |
| `negent`     | semantic-entropy negentropy over the k=10 clustered samples |

**What the demo shows** (all pure, cached computation — **no API keys, no network LLM calls**):

1. **The symbolic engine, live** — the kinship forward-closure certificate runs on *real* extracted
   edges: it **derives** a relation with an auditable trace-graph on a present pair, and **abstains**
   (structurally, hallucination-safe) on an absent pair.
2. **FACT A** — the raw LLM commits a confident kinship on ~33% of *absent* pairs (people in
   *different connected components* of the family graph → no kinship path can exist).
3. **FACT B** — those hallucinations are **invisible to confidence**: they sit at high signal value
   and survive a coverage-matched threshold.
4. **The honest boundary** — on natural prose the certificate *over-abstains* on PRESENT pairs because
   extraction recall is the binding ceiling; we quantify it against the **gold-read ceiling**.

> The certificate logic (`kinship.py`) and the analysis functions (`stats.py`, `baselines.py`,
> `method.py`) are copied **VERBATIM** from the artifact and inlined here so the notebook is
> self-contained. The per-query LLM reads are loaded **pre-computed (cached)** from GitHub — the
> demo reproduces the published numbers without spending a cent.

In [ ]:
# --- Dependency install (works on Colab AND local Jupyter) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# numpy + matplotlib are pre-installed on Colab; behind the google.colab guard they are
# installed LOCALLY at Colab's exact versions (and skipped on Colab to avoid ABI corruption).
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

In [ ]:
# --- Imports (mirrors the artifact's modules) ---
import json, math
from collections import defaultdict, deque

import numpy as np
import matplotlib.pyplot as plt

# numpy 2.0 compat shims (harmless if already present)
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
# --- Data loading: GitHub URL with a local fallback (Colab-compatible) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-40a89b-no-derivation-no-relation-a-closure-cert/main/round-7/experiment-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("Loaded mini demo data:", data["description"][:90], "...")
print("  composition_table relation types:", list(data["composition_table"]["relation_types"].keys()))
print("  worked_cases:", [w["case_type"] for w in data["worked_cases"]])
print("  query_rows:", len(data["query_rows"]),
      "(present=%d absent=%d)" % (
          sum(1 for r in data["query_rows"] if not r["metadata_is_absent"]),
          sum(1 for r in data["query_rows"] if r["metadata_is_absent"])))

## Configuration

All tunable parameters live here. They start at **small** values so the notebook runs end-to-end
quickly; scale them up by editing this one cell.

* `N_QUERY_ROWS` — how many of the loaded per-query rows to analyse (the mini subset holds 100,
  interleaved present/absent so any prefix is a balanced mixed pool).
* `B_BOOT` — doc-clustered paired-bootstrap resamples for the mixed-pool confident-wrong CIs.
  The artifact uses **10000**; a few thousand already gives stable CIs here.

In [ ]:
# ===================== TUNABLE DEMO CONFIG =====================
SEED    = 20260618                                          # RNG seed (matches the artifact)
SIGNALS = ("verbalized", "sc_margin", "ptrue", "negent")   # the 4-signal confidence battery

# --- scale by editing here. To start minimal: N_QUERY_ROWS=20, B_BOOT=200. ---
N_QUERY_ROWS = 100        # full mini subset = 100 rows (the published full corpus has 728)
B_BOOT       = 10000      # artifact value (runs in ~2s here); start small (e.g. 200) to go faster
# ===============================================================
print(f"config: N_QUERY_ROWS={N_QUERY_ROWS}  B_BOOT={B_BOOT}  SIGNALS={SIGNALS}")

## 1 · The symbolic certificate engine (`kinship.py`, VERBATIM)

The kinship calculus is a **finite composition table** over 11 abstract relation types — *not* a full
relation algebra. The sound closure for such a table is a **forward least-fixpoint UNION derivation**
over *defined* compositions only:

* seed every atomic edge `a→b:t` (and its converse `b→a:conv(t)`);
* while changing, for every `a–b–c` with `t1∈D[a,b]`, `t2∈D[b,c]`, if `rules[t1][t2]=t3` is **defined**,
  add `t3` to `D[a,c]`.

**Output contract:** `|D[query]|==1` → emit the relation; `>1` → abstain (conflict); `==0` → **abstain
(no connecting path)**. The last case is the absent-relation regime — with no path the engine
*never invents* a kinship. This is the hallucination-safety guarantee, *by construction*.

The cell below is the engine copied **verbatim** from the artifact's `kinship.py`.

In [ ]:
from __future__ import annotations

from collections import defaultdict, deque
from typing import Iterable


class Kinship:
    """Finite kinship composition calculus parsed from the dataset composition table."""

    def __init__(self, comp_table: dict):
        rt = comp_table["relation_types"]
        self.base: list[str] = list(rt.keys())  # 11 abstract relation types
        self.universe = frozenset(self.base)
        self.empty = frozenset()
        self.symmetric_types = set(comp_table["symmetric_types"])  # {'sibling','SO'}
        self.inv: dict[str, str] = {}
        for a, b in comp_table["inverse_pairs"].items():
            self.inv[a] = b
            self.inv[b] = a
        self.composition_rules = comp_table["composition_rules"]
        self.surface_forms = comp_table["surface_forms"]
        self.surface_reverse = comp_table["surface_reverse"]
        self.label_map = comp_table.get("label_map", {})
        self.label_map_reverse = comp_table.get("label_map_reverse", {})
        # ---- total converse over every base type (sound; no empties) ----
        self._conv: dict[str, str] = {}
        for t in self.base:
            if t in self.symmetric_types:
                self._conv[t] = t
            elif t in self.inv:
                self._conv[t] = self.inv[t]
            elif t == "sibling-in-law":
                # brother/sister-in-law are mutual: converse(sibling-in-law)=sibling-in-law.
                self._conv[t] = t
            else:
                self._conv[t] = t  # sound self-converse fallback (never reached for the 11 types)

    # ------------------------------------------------------------------ ops
    def conv_type(self, t: str) -> str:
        return self._conv[t]

    def compose_types(self, t1: str, t2: str):
        """Defined composition rules[t1][t2]=t3, else None (UNDEFINED == 'unknown')."""
        return self.composition_rules.get(t1, {}).get(t2)

    def label(self, s) -> str:
        s = frozenset(s)
        if not s:
            return "EMPTY"
        if s == self.universe:
            return "UNIVERSE"
        return "|".join(t for t in self.base if t in s)

    # ------------------------------------------------------------- surface words
    def surface(self, rel_type: str, gender: str) -> str:
        g = "male" if str(gender).lower().startswith("m") else "female"
        sf = self.surface_forms.get(rel_type)
        if not sf:
            return rel_type
        return sf.get(g, sf.get("male", rel_type))

    def surface_to_type(self, surface_word: str):
        """Return (relation_type, implied_gender) or None for an unknown word."""
        w = str(surface_word).strip().lower()
        rev = self.surface_reverse.get(w)
        if rev is None:
            return None
        return rev[0], rev[1]


# --------------------------------------------------------------------------- #
# Forward least-fixpoint UNION derivation (the sound closure for the finite table)
# --------------------------------------------------------------------------- #
def _seed(kin: Kinship, atomic_edges: list[dict]):
    """Seed D with atomic edges + their converses. Returns (D, nbrs).
    D[(a,b)] = set of types; nbrs[a] = set of directed successors."""
    D: dict = defaultdict(set)
    nbrs: dict = defaultdict(set)

    def add(a, b, t):
        if t not in D[(a, b)]:
            D[(a, b)].add(t)
            nbrs[a].add(b)

    for e in atomic_edges:
        t = e["type"]
        if t not in kin.base:
            continue
        a, b = e["a"], e["b"]
        if a == b:
            continue
        add(a, b, t)
        add(b, a, kin.conv_type(t))
    return D, nbrs


def forward_closure(kin: Kinship, atomic_edges: list[dict], with_prov: bool = False):
    """Forward least-fixpoint union derivation. Returns (D, nbrs, n_fired) or, with
    with_prov, (D, nbrs, n_fired, prov) where prov[(a,c,t3)] = (a,b,c,t1,t2,t3) records
    the FIRST composition that produced type t3 on pair (a,c) (a directed-edge of the
    proof DAG; seed edges map to None).

    D[(a,b)] holds EVERY relation type derivable for the directed pair a->b; closed
    under defined composition + converse. n_fired = number of new type-additions."""
    D, nbrs = _seed(kin, atomic_edges)
    prov: dict = {}
    if with_prov:
        for (a, b), ts in D.items():
            for t in ts:
                prov.setdefault((a, b, t), None)
    Q = deque(D.keys())
    inq = set(D.keys())
    n_fired = 0

    def push(p):
        if p not in inq:
            inq.add(p)
            Q.append(p)

    def emit(a, c, t3, provtuple):
        nonlocal n_fired
        grew = False
        if t3 not in D[(a, c)]:
            D[(a, c)].add(t3)
            nbrs[a].add(c)
            if with_prov:
                prov.setdefault((a, c, t3), provtuple)
            n_fired += 1
            grew = True
        ct3 = kin.conv_type(t3)
        if ct3 not in D[(c, a)]:
            D[(c, a)].add(ct3)
            nbrs[c].add(a)
            if with_prov:
                prov.setdefault((c, a, ct3), (c, a, a, ct3, None, ct3))  # converse marker
        if grew:
            push((a, c)); push((c, a))

    while Q:
        (a, b) = Q.popleft()
        inq.discard((a, b))
        tab = list(D[(a, b)])
        # extend a->b with b->c  =>  a->c
        for c in list(nbrs[b]):
            if c == a:
                continue
            for t1 in tab:
                for t2 in list(D[(b, c)]):
                    t3 = kin.compose_types(t1, t2)
                    if t3 is not None:
                        emit(a, c, t3, (a, b, c, t1, t2, t3))
        # extend z->a with a->b  =>  z->b   (a is the middle)
        for z in list(nbrs[a]):
            if z == b:
                continue
            for t1 in list(D[(z, a)]):
                for t2 in tab:
                    t3 = kin.compose_types(t1, t2)
                    if t3 is not None:
                        emit(z, b, t3, (z, a, b, t1, t2, t3))
    if with_prov:
        return D, nbrs, n_fired, prov
    return D, nbrs, n_fired


def naive_single_pass(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt) -> set:
    """BASELINE: ONE composition pass at the query edge using ONLY seed (atomic) edges.

    R = union over intermediates w of {rules[t1][t2] : t1 in seed(u,w), t2 in seed(w,v)}.
    NO fixpoint, NO derived edges. On a hop-k chain only the hop-2 case has an
    intermediate w with BOTH atomic links to the endpoints, so naive resolves hop-2 but
    derives nothing (-> abstain) on hop>=3."""
    D, nbrs = _seed(kin, atomic_edges)
    R: set = set()
    for w in nbrs.get(qsrc, ()):
        if w in (qsrc, qtgt):
            continue
        if (w, qtgt) in D:
            for t1 in D[(qsrc, w)]:
                for t2 in D[(w, qtgt)]:
                    t3 = kin.compose_types(t1, t2)
                    if t3 is not None:
                        R.add(t3)
    return R


# --------------------------------------------------------------------------- #
# Query wrappers with the Mode-A / Mode-B output contract
# --------------------------------------------------------------------------- #
def _answer_from_set(kin: Kinship, R: set) -> dict:
    R = set(R)
    n = len(R)
    if n == 1:
        t = next(iter(R))
        return {"types": sorted(R), "singleton": True, "answer_type": t,
                "n_derivations": n, "mode_b_conflict": False, "no_path": False}
    if n == 0:
        return {"types": [], "singleton": False, "answer_type": None,
                "n_derivations": 0, "mode_b_conflict": False, "no_path": True}
    # n > 1 : incompatible derivations => Mode-B conflict
    rep = sorted(R, key=lambda t: kin.base.index(t))[0]  # deterministic representative
    return {"types": sorted(R), "singleton": False, "answer_type": rep,
            "n_derivations": n, "mode_b_conflict": True, "no_path": False}


def query_modeA(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt) -> dict:
    """Mode-A forward-closure query. Returns the output-contract decision + n_fired."""
    D, nbrs, n_fired = forward_closure(kin, atomic_edges)
    R = D.get((qsrc, qtgt), set())
    out = _answer_from_set(kin, R)
    out["n_fired"] = n_fired
    return out


def query_naive(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt) -> dict:
    """Naive single-pass query (fresh seed only)."""
    R = naive_single_pass(kin, atomic_edges, qsrc, qtgt)
    return _answer_from_set(kin, R)


def simple_paths_names(atomic_edges: list[dict], qsrc, qtgt, max_paths: int = 3,
                       max_len: int = 12):
    """Up to `max_paths` simple undirected entity paths qsrc..qtgt over the atomic-edge
    graph (feeds Path-of-Thoughts). Returns lists of node names, shortest first."""
    adj: dict = {}
    for e in atomic_edges:
        adj.setdefault(e["a"], set()).add(e["b"])
        adj.setdefault(e["b"], set()).add(e["a"])
    if qsrc not in adj or qtgt not in adj:
        return []
    paths: list[list] = []
    stack = [(qsrc, [qsrc])]
    while stack and len(paths) < max_paths * 4:
        node, path = stack.pop()
        if len(path) > max_len:
            continue
        for nb in sorted(adj.get(node, ())):
            if nb == qtgt:
                paths.append(path + [nb])
            elif nb not in path:
                stack.append((nb, path + [nb]))
    paths.sort(key=len)
    # de-dup
    seen = set(); uniq = []
    for p in paths:
        k = tuple(p)
        if k not in seen:
            seen.add(k); uniq.append(p)
    return uniq[:max_paths]


def derivation_trace(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt,
                     max_steps: int = 60):
    """Reconstruct ONE concrete derivation for (qsrc->qtgt) for the trace-graph:
    which (t1 o t2 -> t3) compositions fire, mirroring the gold backward proof.
    Returns a list of {a,b,c,t1,t2,t3} steps producing the answer type, or [] if the
    query is not a unique-derivation singleton."""
    D, nbrs, _, prov = forward_closure(kin, atomic_edges, with_prov=True)
    target = D.get((qsrc, qtgt), set())
    if len(target) != 1:
        return []
    goal_type = next(iter(target))
    steps = []
    stack = [(qsrc, qtgt, goal_type)]
    seen = set()
    while stack and len(steps) < max_steps:
        key = stack.pop()
        if key in seen:
            continue
        seen.add(key)
        p = prov.get(key)
        if p is None:
            continue  # seed edge (atomic fact) -- a leaf of the proof DAG
        a, b, c, t1, t2, t3 = p
        if t2 is None:
            # converse marker: unfold to the forward edge (b->a : conv(t3))
            stack.append((c, a, kin.conv_type(t3)))
            continue
        steps.append({"a": a, "b": b, "c": c, "t1": t1, "t2": t2, "t3": t3})
        stack.append((a, b, t1))
        stack.append((b, c, t2))
    steps.reverse()
    return steps


# --- build the calculus from the demo's composition table ---
kin = Kinship(data["composition_table"])
print("relation types:", kin.base)
print("converse map: ", kin._conv)

## 2 · The engine running live on real extracted edges

Three real cases from the corpus (each carries the *actual* LLM-extracted atomic kinship edges):

* **present_deduction** — a multi-hop pair the certificate **resolves**, with the full derivation trace.
* **absent_no_derivation** — two people in different components → empty derivation → **ABSTAIN**, while
  the raw LLM confidently commits a (wrong) relation. This is FACT A + FACT B in a single example.
* **present_over_abstain** — a present pair where extraction *missed* a connecting edge, so the
  LLM-read certificate over-abstains (the gold-read certificate would have solved it). This is the
  extraction-limited boundary in a single example.

In [ ]:
def run_certificate(case):
    edges = [{"a": e["a"], "b": e["b"], "type": e["type"]} for e in case["extracted_atomics"]]
    ans = query_modeA(kin, edges, case["qsrc_name"], case["qtgt_name"])
    if ans["no_path"]:
        decision = "ABSTAIN (no derivation path)"
    elif ans["mode_b_conflict"]:
        decision = "ABSTAIN (conflicting derivations)"
    else:
        decision = f"DERIVE -> {ans['answer_type']}"
    trace = derivation_trace(kin, edges, case["qsrc_name"], case["qtgt_name"])
    return decision, trace

for case in data["worked_cases"]:
    print("=" * 78)
    print(f"[{case['case_type']}]  {case['title']}")
    print("  Q:", case["question"])
    print("  extracted atomic edges:")
    for e in case["extracted_atomics"]:
        print(f"      {e['a']}  --{e['type']}({e['surface']})-->  {e['b']}")
    decision, trace = run_certificate(case)
    print(f"  query: ({case['qsrc_name']}) -> ({case['qtgt_name']})")
    print(f"  GOLD            : {case['gold_word']}  [{case['gold_primitive']}]")
    print(f"  CERTIFICATE     : {decision}   (expected: {case['expected_certificate']})")
    if case.get("raw_llm_committed"):
        sig = case.get("raw_llm_signals") or {}
        sigstr = "  ".join(f"{k}={sig.get(k)}" for k in ("verbalized","sc_margin","ptrue","negent") if k in sig)
        print(f"  RAW LLM committed: {case['raw_llm_committed']}    signals: {sigstr}")
    if trace:
        print("  derivation trace-graph (auditable):")
        for st in trace:
            print(f"      {st['a']} o {st['b']} o {st['c']}:  {st['t1']} . {st['t2']}  ->  {st['t3']}")
print("=" * 78)

## 3 · Reconstruct the per-query records for the population analysis

Each cached row carries the certificate decision, the gold relation, the raw LLM's commitment and
its four confidence signals. We rebuild the lightweight `records` the artifact's analysis functions
expect (one dict per query, with method sub-dicts `modeA` = certificate, `raw`, `ct_<signal>`, …).
Scoring is **primitive-level** (gender is best-effort in DocRED); the human gendered word is kept as
`surface_word`.

In [ ]:
def prim(word):
    """Map a surface kinship word to its gender-independent primitive (None if abstained)."""
    if word is None or word == "ABSTAIN":
        return None
    m = kin.surface_to_type(word)
    return m[0] if m else str(word).lower()

def build_records(rows):
    recs = []
    for e in rows:
        cert_named = e["predict_certificate"] != "ABSTAIN"
        gr_named   = e["predict_certificate_goldread"] != "ABSTAIN"
        raw_named  = bool(e["metadata_raw_named"])
        raw_surf   = prim(e["predict_commit_argmax"]) if raw_named else None
        sig = {s: float(e[f"metadata_conf_{s}"]) for s in SIGNALS}
        r = {
            "is_absent": bool(e["metadata_is_absent"]),
            "gold_surface": e["metadata_gold_primitive"],            # primitive gold
            "gold_surface_word": e["output"],                        # gendered gold word
            "doc_id": e["metadata_doc_id"],
            # the symbolic certificate is certain when it commits (conf 1.0), else ignored
            "modeA":          {"named": cert_named, "conf": 1.0 if cert_named else 0.0,
                               "surface": e["metadata_certificate_primitive"],
                               "surface_word": e["predict_certificate"]},
            "modeA_goldread": {"named": gr_named,   "conf": 1.0 if gr_named else 0.0,
                               "surface": prim(e["predict_certificate_goldread"]),
                               "surface_word": e["predict_certificate_goldread"]},
            "raw":            {"named": raw_named,  "conf": sig["verbalized"], "surface": raw_surf,
                               "surface_word": e["predict_commit_argmax"] if raw_named else None},
            "_sig": sig,
        }
        for s in SIGNALS:
            r[f"ct_{s}"] = {"named": raw_named, "conf": sig[s], "surface": raw_surf}
        recs.append(r)
    return recs

records = build_records(data["query_rows"][:N_QUERY_ROWS])
n_pres = sum(1 for r in records if not r["is_absent"])
n_abs  = sum(1 for r in records if r["is_absent"])
print(f"built {len(records)} records  (present={n_pres}  absent={n_abs})")

## 4 · Analysis helpers (`stats.py` / `baselines.py` / `method.py`, VERBATIM)

These are the artifact's own scoring primitives, inlined unchanged (the only edit is dropping the
`B.` module-prefix now that the helpers live in the same namespace):

* `matched_coverage_mask` — top-k-by-confidence mask at a target coverage (`stats.py`).
* `coverage_confidence`, `confident_wrong` — coverage ranking & the confident-wrong predicate (`baselines.py`).
* `_r`, `_by_doc` — rounding & doc-clustering helpers (`method.py`).

In [ ]:
def matched_coverage_mask(conf: np.ndarray, target_cov: float) -> np.ndarray:
    """Boolean mask covering the top ceil(target_cov * N) networks by confidence.

    Rank-based (handles discrete/degenerate confidences gracefully): ties broken by index
    so the choice is deterministic. Returns an all-False mask if target_cov<=0."""
    n = len(conf)
    k = int(round(target_cov * n))
    k = max(0, min(n, k))
    mask = np.zeros(n, dtype=bool)
    if k == 0:
        return mask
    # argsort by (-conf, index): highest confidence first, stable
    order = sorted(range(n), key=lambda i: (-conf[i], i))
    for i in order[:k]:
        mask[i] = True
    return mask



def coverage_confidence(named: bool, conf: float) -> float:
    """Confidence used for COVERAGE ranking: an ABSTENTION is never 'covered', so its
    coverage-confidence is -1 (below any real [0,1] confidence). This prevents a method's
    confident *abstentions* from filling the coverage budget ahead of its lower-ranked
    *named* answers -- coverage counts only relations the method actually emits."""
    return float(conf) if named else -1.0


def query_correct(named: bool, surface, gold_surface: str, is_absent: bool) -> bool:
    """Correct = (named AND surface==gold) for present queries; (NOT named) for absent."""
    if is_absent:
        return not named
    return bool(named and surface == gold_surface)


def confident_wrong(named: bool, surface, gold_surface: str, is_absent: bool) -> bool:
    """A named answer that disagrees with gold (for absent, ANY named answer is wrong)."""
    if not named:
        return False
    if is_absent:
        return True
    return surface != gold_surface


def _r(x, nd=4):
    try:
        if x != x:
            return float("nan")
        return round(float(x), nd)
    except (TypeError, ValueError):
        return x


def _by_doc(doc_ids):
    d = defaultdict(list)
    for i, x in enumerate(doc_ids):
        d[x].append(i)
    return d

## 5 · FACT A + FACT B — `crux_survival_table` (VERBATIM)

On the absent queries, take the subset where the RAW LLM is confident-wrong (named a relation when
the gold is *no-relation*). For each confidence signal report its **distribution on those
hallucinations** and the fraction that **survive** a single global threshold tuned to the
certificate's coverage.

* **FACT A** = `raw_hallucination_rate_absent` (how often the raw LLM invents a kinship).
* **FACT B** = high mean signal + high `frac_surviving_certificate_matched_rule` (the hallucinations
  are *invisible* to confidence). The certificate, by contrast, abstains on ~all of them structurally
  (`certificate_confident_wrong_absent` ≈ 0).

In [ ]:
def crux_survival_table(records):
    """FACT A + FACT B. On absent queries take the subset where RAW is confident-wrong
    (named a relation, gold=no-relation). Per signal: the signal DISTRIBUTION on those
    hallucinations + the fraction that SURVIVE a confidence rule calibrated to the
    certificate's coverage.

    `records` is the MIXED present+absent pool. The survival threshold is GLOBAL and matched
    to the certificate's MIXED coverage (a single neural threshold must serve present AND
    absent): when the certificate abstains on ~100% of absent pairs its absent-only coverage
    is 0 (degenerate, as the iter-6 README flagged), so the meaningful operating point is the
    global threshold that gives the LLM the certificate's overall coverage -- under it, what
    fraction of the absent hallucinations does the LLM still COMMIT? An absent-only
    calibration is kept as a secondary diagnostic."""
    absent = [r for r in records if r["is_absent"]]
    N = len(records); N_abs = len(absent)
    halluc = [r for r in absent if r["raw"]["named"]]
    n_h = len(halluc)
    cert_named_abs = np.array([r["modeA"]["named"] for r in absent], bool) if N_abs else np.array([], bool)
    cert_cov_abs = float(cert_named_abs.mean()) if N_abs else float("nan")
    cert_cw_abs = float(np.mean([confident_wrong(r["modeA"]["named"], r["modeA"]["surface"],
                                                   r["gold_surface"], True) for r in absent])) if N_abs else float("nan")
    cert_cov_mixed = float(np.mean([1.0 if r["modeA"]["named"] else 0.0 for r in records])) if N else float("nan")
    out = {"n_absent": N_abs, "n_raw_confident_wrong": n_h,
           "raw_hallucination_rate_absent": _r(n_h / N_abs if N_abs else 0.0),
           "certificate_coverage_absent": _r(cert_cov_abs),
           "certificate_coverage_mixed": _r(cert_cov_mixed),
           "certificate_confident_wrong_absent": _r(cert_cw_abs), "per_signal": {}}
    for s in SIGNALS:
        m = f"ct_{s}"
        # GLOBAL threshold over the MIXED pool, matched to the certificate's mixed coverage
        conf_mixed = np.array([coverage_confidence(r[m]["named"], r[m]["conf"]) for r in records], float)
        mask_mix = matched_coverage_mask(conf_mixed, cert_cov_mixed)
        covered_mix = sorted([conf_mixed[i] for i in range(N) if mask_mix[i] and conf_mixed[i] >= 0.0])
        tau_global = covered_mix[0] if covered_mix else float("nan")
        # ABSENT-only calibration (secondary; degenerate when cert abstains on all absent)
        conf_abs = np.array([coverage_confidence(r[m]["named"], r[m]["conf"]) for r in absent], float)
        mask_abs = matched_coverage_mask(conf_abs, cert_cov_abs)
        covered_abs = sorted([conf_abs[i] for i in range(N_abs) if mask_abs[i] and conf_abs[i] >= 0.0])
        tau_abs = covered_abs[0] if covered_abs else float("nan")
        vals = np.array([r["_sig"][s] for r in halluc], float) if n_h else np.array([])
        if n_h:
            q = np.quantile(vals, [0.10, 0.25, 0.50, 0.75, 0.90])
            pool_median = float(np.median([r["_sig"][s] for r in absent]))
            frac_ge_half = float(np.mean(vals >= 0.5))
            frac_ge_poolmed = float(np.mean(vals >= pool_median))
            frac_surv_global = float(np.mean(vals >= tau_global)) if tau_global == tau_global else float("nan")
            frac_surv_abs = float(np.mean(vals >= tau_abs)) if tau_abs == tau_abs else float("nan")
            dist = {"mean": _r(float(vals.mean())), "median": _r(float(q[2])), "p10": _r(float(q[0])),
                    "p25": _r(float(q[1])), "p75": _r(float(q[3])), "p90": _r(float(q[4]))}
        else:
            frac_ge_half = frac_ge_poolmed = frac_surv_global = frac_surv_abs = float("nan"); dist = {}
        out["per_signal"][m] = {
            "tau_global_at_certificate_mixed_coverage": _r(tau_global),
            "tau_absent_only": _r(tau_abs),
            "signal_distribution_on_hallucinations": dist,
            "frac_hallucinations_signal_ge_0.5": _r(frac_ge_half),
            "frac_hallucinations_signal_ge_pool_median": _r(frac_ge_poolmed),
            "frac_surviving_certificate_matched_rule": _r(frac_surv_global),
            "frac_surviving_absent_only_calibration": _r(frac_surv_abs),
            "interpretation": ("frac_surviving_certificate_matched_rule = fraction of the LLM's "
                               "high-confidence absent-relation hallucinations that a SINGLE GLOBAL "
                               "confidence threshold (tuned to the certificate's mixed coverage) would "
                               "still COMMIT. The certificate abstains on ~all of them STRUCTURALLY "
                               "(no derivation path), regardless of confidence."),
        }
    return out


crux = crux_survival_table(records)
print(f"FACT A  raw absent-hallucination rate = {crux['raw_hallucination_rate_absent']}  "
      f"({crux['n_raw_confident_wrong']}/{crux['n_absent']})")
print(f"        certificate confident-wrong on absent = {crux['certificate_confident_wrong_absent']}  "
      f"(structural abstention)")
print()
print("FACT B  per-signal behaviour on the raw LLM's absent hallucinations:")
hdr = f"  {'signal':<12}{'mean':>8}{'frac>=0.5':>11}{'frac_surviving':>16}"
print(hdr); print('  ' + '-'*45)
for s in SIGNALS:
    ps = crux['per_signal'][f'ct_{s}']
    dist = ps['signal_distribution_on_hallucinations']
    print(f"  {s:<12}{dist.get('mean'):>8}{ps['frac_hallucinations_signal_ge_0.5']:>11}"
          f"{ps['frac_surviving_certificate_matched_rule']:>16}")

## 6 · Abstention decomposition + the gold-read ceiling — `abstention_decomposition` (VERBATIM)

The certificate's abstentions split into **correct-absent** (right call: no relation) vs
**over-abstain-present** (missed a connecting edge on a present pair). The **gold-read ceiling** — the
same closure run over the *gold* atomic edges — reproduces 100% of present golds and abstains on 100%
of absent pairs **by construction**. The gap between LLM-read and gold-read present coverage is exactly
the natural-prose **extraction ceiling** — the binding constraint, isolated.

In [ ]:
def abstention_decomposition(records, ref="modeA", gold_ref="modeA_goldread"):
    present = [r for r in records if not r["is_absent"]]
    absent = [r for r in records if r["is_absent"]]
    N = len(records)
    correct_absent = sum(1 for r in absent if not r[ref]["named"])
    over_abstain_present = sum(1 for r in present if not r[ref]["named"])
    named_total = sum(1 for r in records if r[ref]["named"])
    pres_named = [r for r in present if r[ref]["named"]]
    present_coverage = (len(pres_named) / len(present)) if present else float("nan")
    # primitive-level present selective accuracy
    pres_sel_prim = (np.mean([1.0 if r[ref]["surface"] == r["gold_surface"] else 0.0 for r in pres_named])
                     if pres_named else float("nan"))
    # surface-level (secondary; gender best-effort)
    pres_sel_surf = (np.mean([1.0 if (r[ref].get("surface_word") == r["gold_surface_word"]) else 0.0
                              for r in pres_named]) if pres_named else float("nan"))
    # gold-read ceiling (isolates extraction recall as the binding ceiling)
    g_pres_named = [r for r in present if r[gold_ref]["named"]]
    gold_present_coverage = (len(g_pres_named) / len(present)) if present else float("nan")
    gold_correct_absent = sum(1 for r in absent if not r[gold_ref]["named"])
    gold_pres_sel = (np.mean([1.0 if r[gold_ref]["surface"] == r["gold_surface"] else 0.0 for r in g_pres_named])
                     if g_pres_named else float("nan"))
    return {
        "n_total": N, "n_present": len(present), "n_absent": len(absent),
        "certificate_named_total": named_total,
        "correct_absent_abstentions": correct_absent,
        "correct_absent_abstention_rate": _r(correct_absent / len(absent) if absent else float("nan")),
        "over_abstain_present": over_abstain_present,
        "over_abstain_present_rate": _r(over_abstain_present / len(present) if present else float("nan")),
        "present_coverage_llm_read": _r(present_coverage),
        "present_selective_accuracy_primitive": _r(float(pres_sel_prim)),
        "present_selective_accuracy_surface": _r(float(pres_sel_surf)),
        "gold_read_ceiling": {
            "present_coverage": _r(gold_present_coverage),
            "correct_absent_abstention_rate": _r(gold_correct_absent / len(absent) if absent else float("nan")),
            "present_selective_accuracy_primitive": _r(float(gold_pres_sel)),
            "note": ("the gold-read certificate (closure over GOLD atomic edges) reproduces 100% of "
                     "present golds & abstains on 100% of absent pairs by construction; the gap to the "
                     "LLM-read certificate is exactly the natural-prose EXTRACTION ceiling.")},
        "interpretation": ("On natural prose the extracted graph is no longer trivially complete, so the "
                           "certificate can OVER-ABSTAIN on PRESENT pairs (missing connecting edges look "
                           "disconnected). correct_absent + over_abstain_present + named == total decisions; "
                           "high over_abstain_present with low present_coverage => extraction-limited."),
    }


decomp = abstention_decomposition(records)
print(f"present pairs         : {decomp['n_present']}    absent pairs: {decomp['n_absent']}")
print(f"correct-absent rate   : {decomp['correct_absent_abstention_rate']}  (structural, hallucination-safe)")
print(f"over-abstain-present  : {decomp['over_abstain_present_rate']}  (extraction recall miss)")
print(f"present coverage      : LLM-read {decomp['present_coverage_llm_read']}  "
      f"vs GOLD-read {decomp['gold_read_ceiling']['present_coverage']} (ceiling)")
print(f"present selective-acc : LLM-read {decomp['present_selective_accuracy_primitive']}  "
      f"vs GOLD-read {decomp['gold_read_ceiling']['present_selective_accuracy_primitive']} (ceiling)")

## 7 · The mixed-pool showdown — `cw_matched_to_ref` (VERBATIM)

The decisive test: on the MIXED present+absent pool, is the certificate's **confident-wrong** rate
lower than each confidence signal's, at matched coverage? Reductions are measured with a
**doc-clustered paired bootstrap** (`B_BOOT` resamples). On natural prose these come out slightly
**negative with CIs spanning 0** — the certificate does *not* win the mixed pool, because extraction
noise propagates to wrong derivations and over-abstention. That is the pre-registered
**EXTRACTION-LIMITED-BOUNDARY** verdict, quantified rather than asserted.

In [ ]:
def cw_matched_to_ref(records, ref, compare, n_boot=B_BOOT, seed=SEED):
    """Confident-wrong reduction (compare - ref) at coverage matched to the REFERENCE's
    natural coverage; story/doc-clustered paired bootstrap. Decisive on the MIXED pool."""
    recs = records
    N = len(recs)
    if N == 0:
        return {"n": 0}
    doc_ids = [r["doc_id"] for r in recs]
    conf = {m: np.array([coverage_confidence(r[m]["named"], r[m]["conf"]) for r in recs], float)
            for m in (ref, compare)}
    cw = {m: np.array([confident_wrong(r[m]["named"], r[m]["surface"], r["gold_surface"], r["is_absent"])
                       for r in recs], float) for m in (ref, compare)}
    named_ref = np.array([r[ref]["named"] for r in recs], bool)
    c_match = float(named_ref.mean())
    mask_ref = named_ref
    mask_cmp = matched_coverage_mask(conf[compare], c_match)
    ref_rate = float((cw[ref] * mask_ref).sum() / N)
    cmp_rate = float((cw[compare] * mask_cmp).sum() / N)
    by_doc = _by_doc(doc_ids); docs = list(by_doc); nd = len(docs)
    rng = np.random.default_rng(seed)
    cwr, cwc = cw[ref], cw[compare]
    diffs = []
    for _ in range(n_boot):
        pick = rng.integers(0, nd, nd)
        idx = np.concatenate([by_doc[docs[i]] for i in pick])
        n = len(idx)
        d_ref = float((cwr[idx] * mask_ref[idx]).sum() / n)
        d_cmp = float((cwc[idx] * mask_cmp[idx]).sum() / n)
        diffs.append(d_cmp - d_ref)
    diffs = np.array(diffs, float)
    lo, hi = np.quantile(diffs, [0.025, 0.975])
    p_one = max(float(np.mean(diffs <= 0.0)), 1.0 / (len(diffs) + 1))
    return {"n": N, "matched_coverage": _r(c_match),
            "ref_confident_wrong": _r(ref_rate), "compare_confident_wrong": _r(cmp_rate),
            "confident_wrong_reduction": _r(cmp_rate - ref_rate),
            "ci95": [_r(lo), _r(hi)], "p_one_sided": _r(p_one), "ci_excludes_0": bool(lo > 0.0)}


print(f"mixed-pool confident-wrong reduction (certificate - signal), B_BOOT={B_BOOT}:")
print(f"  {'signal':<12}{'reduction':>11}{'95% CI':>22}{'excl 0?':>9}")
print('  ' + '-'*53)
showdown = {}
for s in SIGNALS:
    res = cw_matched_to_ref(records, ref='modeA', compare=f'ct_{s}', n_boot=B_BOOT, seed=SEED)
    showdown[s] = res
    ci = res['ci95']
    print(f"  {s:<12}{res['confident_wrong_reduction']:>11}"
          f"{('['+str(ci[0])+', '+str(ci[1])+']'):>22}{str(res['ci_excludes_0']):>9}")
print(f"\n  matched coverage = {showdown['verbalized']['matched_coverage']}  "
      f"(certificate's natural coverage on the mixed pool)")

## 8 · Visual summary

* **Left** — FACT B: mean signal value on the raw LLM's absent hallucinations. Bars near 1.0 mean the
  signal is *blind* to the hallucination (P(True) is the only partial discriminator).
* **Middle** — confident-wrong on the absent pool: the certificate (structural abstention) vs the raw
  LLM committing under each signal.
* **Right** — present coverage: LLM-read certificate vs the gold-read ceiling (the extraction gap).

The printed block compares the **demo-subset** numbers against the **published full-corpus** headline.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) FACT B: mean signal on hallucinations
means = [crux['per_signal'][f'ct_{s}']['signal_distribution_on_hallucinations'].get('mean', 0.0) for s in SIGNALS]
axes[0].bar(SIGNALS, means, color=["#c0392b","#e67e22","#2980b9","#8e44ad"])
axes[0].axhline(0.5, ls="--", c="gray", lw=1)
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("mean signal on hallucinations")
axes[0].set_title("FACT B — confidence is blind\n(higher = more blind)")
for i, v in enumerate(means): axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

# (b) confident-wrong on the ABSENT pool: certificate vs raw-LLM-under-each-signal
labels = ["certificate"] + list(SIGNALS)
cw_cert = crux['certificate_confident_wrong_absent']
# on absent, any named commitment is wrong; the raw LLM commits at the hallucination rate
cw_raw  = crux['raw_hallucination_rate_absent']
vals = [cw_cert] + [cw_raw] * len(SIGNALS)
colors = ["#27ae60"] + ["#c0392b"] * len(SIGNALS)
axes[1].bar(labels, vals, color=colors)
axes[1].set_ylim(0, max(vals) * 1.25 + 0.01); axes[1].set_ylabel("confident-wrong rate (absent pool)")
axes[1].set_title("Absent pool: certificate abstains,\nthe raw LLM commits")
axes[1].tick_params(axis="x", rotation=20)
for i, v in enumerate(vals): axes[1].text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)

# (c) present coverage: LLM-read vs gold-read ceiling
cov = [decomp['present_coverage_llm_read'], decomp['gold_read_ceiling']['present_coverage']]
axes[2].bar(["LLM-read", "gold-read\n(ceiling)"], cov, color=["#2980b9", "#16a085"])
axes[2].set_ylim(0, 1.05); axes[2].set_ylabel("present coverage")
axes[2].set_title("Extraction is the binding ceiling")
for i, v in enumerate(cov): axes[2].text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

plt.tight_layout(); plt.show()

# ---- demo-subset vs published full-corpus headline ----
pub = data["published_headline"]
print("=" * 70)
print("DEMO SUBSET  vs  PUBLISHED FULL-CORPUS HEADLINE")
print("  slice:", pub["slice"])
print("-" * 70)
print(f"{'metric':<38}{'demo':>14}{'published':>16}")
print(f"{'FACT A raw absent-hallucination rate':<38}{crux['raw_hallucination_rate_absent']:>14}"
      f"{pub['FACT_A_raw_absent_hallucination_rate']:>16}")
print(f"{'certificate confident-wrong (absent)':<38}{crux['certificate_confident_wrong_absent']:>14}"
      f"{pub['certificate_absent_confident_wrong']:>16}")
for s in SIGNALS:
    demo_m = crux['per_signal'][f'ct_{s}']['signal_distribution_on_hallucinations'].get('mean')
    print(f"{'  mean ' + s + ' on hallucinations':<38}{demo_m:>14}"
          f"{pub['FACT_B_mean_signal_on_hallucinations']['ct_'+s]:>16}")
print(f"{'present coverage (LLM-read)':<38}{decomp['present_coverage_llm_read']:>14}"
      f"{pub['present_coverage_llm_read']:>16}")
print(f"{'present coverage (gold-read ceiling)':<38}{decomp['gold_read_ceiling']['present_coverage']:>14}"
      f"{pub['gold_read_present_coverage']:>16}")
print("=" * 70)
print("PRE-REGISTERED FORK VERDICT:", pub["fork_verdict"])
print("  FACT A + FACT B transfer to natural prose AND across readers; the certificate stays")
print("  hallucination-safe on absent pairs, but its mixed-pool advantage is erased by extraction")
print("  RECALL (not by the certificate's logic) — the gold-read ceiling isolates that exactly.")